# Bonus: Is There New Physics Hiding? (NSI)

The Day 4 notebook let you adjust θ₂₃, Δm²₃₂, and δ_CP — the standard three-flavor
picture. That picture fits real data very well. But "fits well" is not the same as
"is the whole story." Physicists constantly ask: *is there a small effect the
standard picture is missing?*

One candidate is **NSI (Non-Standard Interactions)**: the idea that neutrinos might
feel extra, not-yet-discovered forces as they pass through matter, on top of the
ordinary weak interaction. If NSI exists, it would nudge the oscillation
probabilities away from the standard prediction — usually by a small amount, which
is exactly why it is hard to find.

You will not derive NSI here. You will do what an experimentalist does first:
turn the new-physics knob on, and see whether the effect is big enough that a real
detector could plausibly notice it.

**This notebook is optional.** It picks up right where Day 4 left off, using the
same `nuosclab` tools you already know.

First time using `nuosclab`? Part 1's notebook has a short section, "A direct nuosclab call, before we wrap it," that walks through the raw API.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from nuosclab import ExplorerConfig, PMNSParams, NSIParams, compute_curves

plt.style.use("seaborn-v0_8-whitegrid")

STANDARD_PMNS = PMNSParams()

## 1. One new-physics knob at a time

NSI has several possible parameters. To keep this manageable, we will only turn on
one: **ε_eμ**, an amplitude, and its phase **δ_eμ**. Setting ε_eμ = 0 reproduces the
standard prediction exactly — that is your built-in sanity check.


In [ ]:
def nsi_curves(experiment="NOvA", eps_emu=0.0, delta_emu_deg=0.0, n_points=700):
    """Compute live (NSI-on) and standard (NSI=0) appearance curves."""
    nsi = NSIParams(eps_emu=eps_emu, delta_emu=np.radians(delta_emu_deg))
    config = ExplorerConfig(
        experiment=experiment,
        pmns=STANDARD_PMNS,
        nsi=nsi,
        n_points=n_points,
        include_standard=True,  # NSI=0 reference curve, computed alongside
        include_nominal=False,
    )
    return compute_curves(config)


def plot_nsi_appearance(experiment="NOvA", eps_emu=0.0, delta_emu_deg=0.0):
    curves = nsi_curves(experiment, eps_emu, delta_emu_deg)
    energy = curves.energy_gev
    appearance_live = curves.live[:, 0, 1]
    appearance_standard = curves.standard[:, 0, 1]

    fig, ax = plt.subplots(figsize=(9, 4.8), constrained_layout=True)
    ax.plot(
        energy,
        appearance_standard,
        color="0.4",
        linestyle="--",
        label=r"standard (NSI = 0)",
    )
    ax.plot(
        energy,
        appearance_live,
        color="#e25c2a",
        linewidth=2,
        label=rf"NSI on: $|\varepsilon_{{e\mu}}|$ = {eps_emu:.3f}",
    )
    ax.axvline(curves.preset.E_peak, color="0.6", linestyle=":", label="beam peak")
    ax.set_title(f"{experiment}: standard prediction vs. NSI-modified appearance")
    ax.set_xlabel("Neutrino energy (GeV)")
    ax.set_ylabel(r"$P(\nu_\mu \to \nu_e)$")
    ax.set_ylim(0, 0.2)
    ax.legend()
    plt.show()


widgets.interact(
    plot_nsi_appearance,
    experiment=widgets.ToggleButtons(options=["NOvA", "DUNE", "T2K"], value="NOvA"),
    eps_emu=widgets.FloatSlider(
        value=0.0, min=0.0, max=0.3, step=0.005, description="|ε_eμ|"
    ),
    delta_emu_deg=widgets.FloatSlider(
        value=0.0, min=-180.0, max=180.0, step=5.0, description="δ_eμ (deg)"
    ),
)

## 2. Exercises

1. Start at ε_eμ = 0. Confirm the solid and dashed curves overlap exactly — this is
   your sanity check that "NSI off" really means "standard physics."
2. Slowly increase ε_eμ. At roughly what value does the solid curve visibly separate
   from the dashed reference curve near the beam peak?
3. In `neutrino_part2_parameter_explorer.ipynb`, the toy near-detector event
   count was a few thousand events. An effect has to be bigger than the toy
   statistical scatter to be "discoverable." Looking at your NOvA plot, does an
   ε_eμ around 0.05 look like something a few-thousand-event toy experiment could
   actually distinguish from the standard curve, or is it too small?
4. Try the same ε_eμ value on NOvA versus DUNE. Which one shows a bigger separation
   from the standard curve? What is different about DUNE (baseline, energy range)
   that might make it more sensitive to new physics?
5. Change δ_eμ instead of ε_eμ. Does the *shape* of the curve change, or mainly its
   *height*? How is that different from what ε_eμ does?


## 3. Why this matters

Real NSI searches work exactly like this exercise, just with real data instead of a
toy curve: compute the standard prediction, compute the NSI-modified prediction,
and ask whether the difference is bigger than the experiment's uncertainty. No
experiment has found a significant NSI effect yet — current data constrain ε_eμ to
be small. But "small" is not "zero," and closing that gap further is one of the
things next-generation experiments like DUNE are designed to do.

This is the same logic used throughout physics: standard model works, but keep
checking the cracks.
